In [64]:
import numpy as np
from scipy import odr as mmq
import pandas as pd
import csv

# Tratamento de Dados Para o Experimento
## Descrição
Lê os arquivos csv gerados pelo pyphox e retorna um novo arquivo com os dados separados e melhor organizados, encontrando os pontos das medidas, obtendo suas médias e intervalos de erro através do desvio padrão da média.

## Recomendação
Esse caderno lê e processa os dados experimentais obtidos através do método de experimento composto, utilizando ambos o sensor de luz e o magnetômetro. Cada medida é considerada como sendo o intervalo em que a luz captada pelo sensor foi bloqueada, os intervalos em que a luz chega ao sensor sem restrições sao considerados como os momentos em que o celular estava sendo movido.

In [ ]:
#------------------------------------------------------------
# 1. Leitura dos Arquivos
#------------------------------------------------------------
# Arquivos de luz e campo magnético na menor altura do trilho entre as bobinas
arquivo_luz_baixo = 'luz_baixo.csv'
arquivo_campo_baixo = 'campo_baixo.csv'

# Arquivos de luz e campo magnético na altura média do trilho entre as bobinas
arquivo_luz_meio = 'luz_meio.csv'
arquivo_campo_meio = 'campo_meio.csv'

# Arquivos de luz e campo magnético na maior altura do trilho entre as bobinas
arquivo_luz_alto = 'luz_alto.csv'
arquivo_campo_alto = 'campo_alto.csv'

In [66]:
# Processamento dos dados
# 1.a. Dados da luz no nível baixo
try:
    df_exp_luz_baixo = pd.read_csv(arquivo_luz_baixo)
    

    # Validação dos dados da luz
    colunas_luz = ["Time (s)", "Illuminance (lx)"]
    for c in colunas_luz:
        if c not in df_exp_luz_baixo:
            raise ValueError(f"Coluna '{c}' não encontrada!")
    
    # Extração da luz
    t_luz_baixo = df_exp_luz_baixo["Time (s)"].values.astype(float).tolist()
    luz_baixo = df_exp_luz_baixo["Illuminance (lx)"].values.astype(float).tolist()

except FileNotFoundError:
    print(f"Arquivo '{arquivo_luz_baixo}' não encontrado!")
except Exception as e:
    print(f"Erro ao ler '{arquivo_luz_baixo}': {e}")

#1.b. Dados do campo magnético no nível baixo
try:
    df_exp_campo_baixo = pd.read_csv(arquivo_campo_baixo)
    
    # Validação dos dados do campo
    colunas_campo = ["Time (s)", "Magnetic field x (µT)", "Magnetic field y (µT)", "Magnetic field z (µT)"]
    for c in colunas_campo:
        if c not in df_exp_campo_baixo:
            raise ValueError(f"Coluna '{c}' não encontrada!")
    
    # Extração do campo
    t_campo_baixo = df_exp_campo_baixo["Time (s)"].values
    bx_baixo = df_exp_campo_baixo["Magnetic field x (µT)"].values.astype(float).tolist()
    by_baixo = df_exp_campo_baixo["Magnetic field y (µT)"].values.astype(float).tolist()
    bz_baixo = df_exp_campo_baixo["Magnetic field z (µT)"].values.astype(float).tolist()
except FileNotFoundError:
    print(f"Arquivo '{arquivo_campo_baixo}' não encontrado!")
except Exception as e:
    print(f"Erro ao ler '{arquivo_campo_baixo}': {e}")

# 2.a. Dados da luz no nível médio
try:
    df_exp_luz_meio = pd.read_csv(arquivo_luz_meio)
    
    # Validação dos dados da luz
    for c in colunas_luz:
        if c not in df_exp_luz_meio:
            raise ValueError(f"Coluna '{c}' não encontrada!")
    
    # Extração da luz
    t_luz_meio = df_exp_luz_meio["Time (s)"].values.astype(float).tolist()
    luz_meio = df_exp_luz_meio["Illuminance (lx)"].values.astype(float).tolist()

except FileNotFoundError:
    print(f"Arquivo '{arquivo_luz_meio}' não encontrado!")
except Exception as e:
    print(f"Erro ao ler '{arquivo_luz_meio}': {e}")

# 2.b. Dados do campo magnético no nível médio
try:
    df_exp_campo_meio = pd.read_csv(arquivo_campo_meio)
    
    # Validação dos dados do campo
    for c in colunas_campo:
        if c not in df_exp_campo_meio:
            raise ValueError(f"Coluna '{c}' não encontrada!")
    
    # Extração do campo
    t_campo_meio = df_exp_campo_meio["Time (s)"].values.astype(float).tolist()
    bx_meio = df_exp_campo_meio["Magnetic field x (µT)"].values.astype(float).tolist()
    by_meio = df_exp_campo_meio["Magnetic field y (µT)"].values.astype(float).tolist()
    bz_meio = df_exp_campo_meio["Magnetic field z (µT)"].values.astype(float).tolist()
except FileNotFoundError:
    print(f"Arquivo '{arquivo_campo_meio}' não encontrado!")
except Exception as e:
    print(f"Erro ao ler '{arquivo_campo_meio}': {e}")

# 3.a. Dados da luz no nível alto
try:
    df_exp_luz_alto = pd.read_csv(arquivo_luz_alto)
    
    # Validação dos dados da luz
    for c in colunas_luz:
        if c not in df_exp_luz_alto:
            raise ValueError(f"Coluna '{c}' não encontrada!")
    
    # Extração da luz
    t_luz_alto = df_exp_luz_alto["Time (s)"].values.astype(float).tolist()
    luz_alto = df_exp_luz_alto["Illuminance (lx)"].values.astype(float).tolist()
except FileNotFoundError:
    print(f"Arquivo '{arquivo_luz_alto}' não encontrado!")
except Exception as e:
    print(f"Erro ao ler '{arquivo_luz_alto}': {e}")

# 3.b. Dados do campo magnético no nível alto
try:
    df_exp_campo_alto = pd.read_csv(arquivo_campo_alto)
    
    # Validação dos dados do campo
    for c in colunas_campo:
        if c not in df_exp_campo_alto:
            raise ValueError(f"Coluna '{c}' não encontrada!")
    
    # Extração do campo
    t_campo_alto = df_exp_campo_alto["Time (s)"].values.astype(float).tolist()
    bx_alto = df_exp_campo_alto["Magnetic field x (µT)"].values.astype(float).tolist()
    by_alto = df_exp_campo_alto["Magnetic field y (µT)"].values.astype(float).tolist()
    bz_alto = df_exp_campo_alto["Magnetic field z (µT)"].values.astype(float).tolist()
except FileNotFoundError:
    print(f"Arquivo '{arquivo_campo_alto}' não encontrado!")
except Exception as e:
    print(f"Erro ao ler '{arquivo_campo_alto}': {e}")

## Determinação das medidas de campo magnético
Considera-se que toda vez que o sensor de luz sofreu uma queda na recepção de luz, uma medida foi tomada.

A média das medidas é calculada para cada um desses intervalos e servirão como os pontos das medidas. Os respectivos desvios padrão são calculados e servirão como a incerteza aproximada das medidas.

In [ ]:
corte = 100 # Limiar para identificar os pontos de interesse (luz < 100 lx), definido com base no comportamento dos dados de luz, mude se necessário

n = 0
somax = 0
somay = 0
somaz = 0

# 1. Cálculo dos pontos médios e desvios padrão para o nível baixo

# Pontos médios
pts_bx_baixo = []
pts_by_baixo = []
pts_bz_baixo = []

# Desvios padrão
dp_pts_bx_baixo = []
dp_pts_by_baixo = []
dp_pts_bz_baixo = []

for i in range(len(luz_baixo)):
    if luz_baixo[i] < corte:
        somax += bx_baixo[i]
        somay += by_baixo[i]
        somaz += bz_baixo[i]
        n += 1
    else:
        if n > 0:
            pts_bx_baixo.append(somax / n)
            dp_pts_bx_baixo.append(float(np.std(bx_baixo[i-n:i])))
            pts_by_baixo.append(somay / n)
            dp_pts_by_baixo.append(float(np.std(by_baixo[i-n:i])))
            pts_bz_baixo.append(somaz / n)
            dp_pts_bz_baixo.append(float(np.std(bz_baixo[i-n:i])))
            n = 0
            somax = 0
            somay = 0
            somaz = 0

# 2. Cálculo dos pontos médios e desvios padrão para o nível médio
# Pontos médios
pts_bx_meio = []
pts_by_meio = []
pts_bz_meio = []

# Desvios padrão
dp_pts_bx_meio = []
dp_pts_by_meio = []
dp_pts_bz_meio = []

for i in range(len(luz_meio)):
    if luz_meio[i] < corte:
        somax += bx_meio[i]
        somay += by_meio[i]
        somaz += bz_meio[i]
        n += 1
    else:
        if n > 0:
            pts_bx_meio.append(somax / n)
            dp_pts_bx_meio.append(float(np.std(bx_meio[i-n:i])))
            pts_by_meio.append(somay / n)
            dp_pts_by_meio.append(float(np.std(by_meio[i-n:i])))
            pts_bz_meio.append(somaz / n)
            dp_pts_bz_meio.append(float(np.std(bz_meio[i-n:i])))
            n = 0
            somax = 0
            somay = 0
            somaz = 0

# 3. Cálculo dos pontos médios e desvios padrão para o nível alto
# Pontos médios
pts_bx_alto = []
pts_by_alto = []
pts_bz_alto = []

# Desvios padrão
dp_pts_bx_alto = []
dp_pts_by_alto = []
dp_pts_bz_alto = []

for i in range(len(luz_alto)):
    if luz_alto[i] < corte:
        somax += bx_alto[i]
        somay += by_alto[i]
        somaz += bz_alto[i]
        n += 1
    else:
        if n > 0:
            pts_bx_alto.append(somax / n)
            dp_pts_bx_alto.append(float(np.std(bx_alto[i-n:i])))
            pts_by_alto.append(somay / n)
            dp_pts_by_alto.append(float(np.std(by_alto[i-n:i])))
            pts_bz_alto.append(somaz / n)
            dp_pts_bz_alto.append(float(np.std(bz_alto[i-n:i])))
            n = 0
            somax = 0
            somay = 0
            somaz = 0

## Lista das posições (cm) nas quais as medidas foram tomadas e propagação do erro
Cria o array com as posições em que as medidas foram tomadas ao longo do trilho entre as bobinas de Helmholtz. Foram traçadas gradações de 1 cm no trilho com uma régua e as medidas foram tomadas em intervalos constantes de distância, com a primeira medida estando sempre na posição definida como 0 cm. A régua possui uma incerteza de metade da menor medida (0,05 cm) e há também a incerteza da posição do magnetômetro em relação a borda superior do celular. 

In [ ]:
# Posições das medidas
positions_baixo = []
positions_meio = []
positions_alto = []

# Incertezas das posições
err_pos_baixo = []
err_pos_meio = []
err_pos_alto = []

# Incertezas da regua e do magnetômetro, mude conforme necessário
err_regua = 0.05
err_mag = 0.6

# Incerteza total das posições por propagação de erros, considerando a incerteza da régua e do magnetômetro
err_pos = np.sqrt(err_regua**2 + err_mag**2)

# Marcadores da posição
j = 0
k = 0
l = 0

for i in range(0, len(pts_bx_baixo)):
    positions_baixo.append(j)
    err_pos_baixo.append(err_pos)
    j += 2 # As medidas foram feitas a cada 2 cm, mude se necessário

for i in range(0, len(pts_bx_meio)):
    positions_meio.append(k)
    err_pos_meio.append(err_pos)
    k += 2 # As medidas foram feitas a cada 2 cm, mude se necessário

for i in range(0, len(pts_bx_alto)):
    positions_alto.append(l)
    err_pos_alto.append(err_pos)
    l += 2 # As medidas foram feitas a cada 2 cm, mude se necessário

## Criação dos Novos arquivos csv
Novos arquivos csv criados para as medidas de campo magnético com os dados já trados para serem lidos pelo simulador. Mude a quantidade de aqrquivos de acordo com o método que utilizou no seu experimento.

In [ ]:
# Exportação dos dados para arquivos CSV
# 1. Exportação dos dados do campo magnético para a menor altura do trilho entre as bobinas
with open('campo_magnetico_baixo.csv', mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['Posição (cm)', 'Incerteza da Posição (cm)', 'Campo Magnético X (µT)', 'Incerteza do Campo X (µT)', 'Campo Magnético Y (µT)', 'Incerteza do Campo Y (µT)', 'Campo Magnético Z (µT)', 'Incerteza do Campo Z (µT)'])
    for i in range(len(pts_bx_baixo)):
        writer.writerow([positions_baixo[i], err_pos_baixo[i], pts_bx_baixo[i], dp_pts_bx_baixo[i], pts_by_baixo[i], dp_pts_by_baixo[i], pts_bz_baixo[i], dp_pts_bz_baixo[i]])

# 2. Exportação dos dados do campo magnético para a altura média do trilho entre as bobinas
with open('campo_magnetico_meio.csv', mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['Posição (cm)', 'Incerteza da Posição (cm)', 'Campo Magnético X (µT)', 'Incerteza do Campo X (µT)', 'Campo Magnético Y (µT)', 'Incerteza do Campo Y (µT)', 'Campo Magnético Z (µT)', 'Incerteza do Campo Z (µT)'])
    for i in range(len(pts_bx_meio)):
        writer.writerow([positions_meio[i], err_pos_meio[i], pts_bx_meio[i], dp_pts_bx_meio[i], pts_by_meio[i], dp_pts_by_meio[i], pts_bz_meio[i], dp_pts_bz_meio[i]])

# 3. Exportação dos dados do campo magnético para a maior altura do trilho entre as bobinas
with open('campo_magnetico_alto.csv', mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['Posição (cm)', 'Incerteza da Posição (cm)', 'Campo Magnético X (µT)', 'Incerteza do Campo X (µT)', 'Campo Magnético Y (µT)', 'Incerteza do Campo Y (µT)', 'Campo Magnético Z (µT)', 'Incerteza do Campo Z (µT)'])
    for i in range(len(pts_bx_alto)):
        writer.writerow([positions_alto[i], err_pos_alto[i], pts_bx_alto[i], dp_pts_bx_alto[i], pts_by_alto[i], dp_pts_by_alto[i], pts_bz_alto[i], dp_pts_bz_alto[i]])